# Document/Image Summarizer + QA Chatbot


Objective:
- Upload a **PDF** or an **image** (`.jpg/.jpeg/.png`) and build a context-aware chatbot that can:
  - **Summarize** the uploaded content
  - Answer questions grounded in that content with citations

Business Use Cases
- Employee self-service support: query policy documents or screenshots/scans of documents without manual search.
- Operations support: extract and summarize information from images (e.g., tables, forms) via OCR.

ML Framing:
- Type: Information Retrieval + Generative NLP (Retrieval-Augmented Generation)
- Input: User query + retrieved policy text chunks
- Output: Natural language answer grounded from the documents and source citations such as page number
- Constraints: Answer only from retrieved documents; refuse if information is unavailable

Note: 
- The chatbot is designed to answer only from retrieved content in the uploaded **file** (PDF or image OCR).
- It will explicitly refuse to answer when the information is not present in the retrieved context.

### Architecture Overview

1. File ingestion
   - PDF: `PyPDFLoader`
   - Images: OCR via `easyocr`
2. Chunking + embeddings (OpenAI embeddings)
3. Vector storage + persistence (ChromaDB)
4. Retrieval of top-k relevant chunks
5. LLM answer + **citations** (and a **summary** right after indexing)

### Preprocessing

1. Import libraries + pipeline helpers
2. Load OpenAI API key
3. Configure caching directory (`chroma_store`)
4. (Optional) Pick a local test file path (PDF/JPG/JPEG/PNG)

1) Import libraries

We’ll persist vector stores so indexing happens only once per document.


In [3]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import os
from pathlib import Path

# Ensure imports resolve from repo root
os.chdir(r"C:\\Users\\ashle\\Project\\usecase")

from src.functions.chatbot_pipeline_extend import (
    PolicyQAConfig,
    load_or_build_vectordb_for_upload,
    answer_question,
    format_context,
    build_demo,
 )

2. Load OpenAI API key in .env file and load .env

In [2]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var. Set it and restart kernel."

3. Set constants

In [3]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4

LLM_MODEL = "gpt-4o-mini"
EMB_MODEL = "text-embedding-3-small"

3. Configure caching (`chroma_store`)
- `chroma_store` is a local vector index cache where Chroma saves embeddings for each uploaded file.
- Normal run (time costly): load file -> split into chunks -> embed chunks -> build Chroma
- With caching, the next run can reuse the saved vector DB for the same file hash.

In [4]:
# Where Chroma persists vector stores (cache)
PERSIST_BASE = Path("chroma_store")
PERSIST_BASE.mkdir(exist_ok=True)

# Let the pipeline use this location by default
os.environ["POLICY_QA_PERSIST_DIR"] = str(PERSIST_BASE.resolve())

In [5]:
# No custom persist_dir helper needed here; the pipeline reads POLICY_QA_PERSIST_DIR.

4. (Optional) Set path for a local test file

Pick either a PDF or an image. This is only used for the programmatic examples below — the Streamlit app lets you upload interactively.

In [6]:
# Default (PDF) - good for QA examples
FILE_PATH = Path(r"data\2025-Vistra-Code-of-Conduct.pdf")

# Optional (image OCR) - uncomment to try OCR
# FILE_PATH = Path(r"data\classification_confusionmatrix_table.png")

assert FILE_PATH.exists(), f"File not found: {FILE_PATH}"

### Vector Store Setup (Pipeline)

This notebook uses the helpers from `src.functions.chatbot_pipeline_extend` to:
- Load a PDF or OCR an image
- Chunk + embed
- Persist a Chroma vector store keyed by file hash

Indexing behavior:
- A stable cache key is computed from the uploaded file bytes (SHA-256).
- Re-indexing the same file becomes instant by reusing the persisted Chroma store.

In [7]:
# Build (or load) the vector DB for the local test file
config = PolicyQAConfig()
doc_id, vectordb = load_or_build_vectordb_for_upload(
    FILE_PATH,
    persist_base=PERSIST_BASE,
    config=config,
)
print("Indexed:", FILE_PATH.name, "doc_id:", doc_id[:12])

Indexed: 2025-Vistra-Code-of-Conduct.pdf doc_id: a64a3aa5ac56


(Optional) Inspect retrieved chunks

This helps confirm retrieval is returning relevant snippets + metadata before generating an answer.

In [8]:
# Loading is handled by the pipeline:
# - PDFs via PyPDFLoader
# - Images via easyocr OCR
# See: src.functions.chatbot_pipeline_extend.load_uploaded_file

3. Chunking

In [9]:
# Chunking is handled by the pipeline (RecursiveCharacterTextSplitter).

4. Embedding

In [10]:
# Embeddings are handled by the pipeline (OpenAIEmbeddings).

5. Build or load the vector store

In [11]:
# Vector store build/load is handled by: load_or_build_vectordb_for_upload(FILE_PATH, ...)

### QA + Summarization

The pipeline provides:
- a **summary** prompt after indexing (in the UI)
- QA with retrieval + citations (RAG)

(Pipeline) Prompting + citations

The system rules and prompt template live in `src.functions.chatbot_pipeline_extend` to keep behavior consistent between notebook and UI.

In [12]:
# Prompt + citation rules are defined in src.functions.chatbot_pipeline_extend (SYSTEM_RULES + PROMPT).

(Pipeline) LLM + embeddings

The pipeline config controls the LLM + embeddings models via `PolicyQAConfig`.

In [13]:
# LLM is created inside the pipeline via get_llm(config=...).

3. Set context metadata
- Include page metadata for the model to cite

In [14]:
# Context formatting for citations is handled by format_context(docs) from the pipeline.

4. Question-Answer model
- Retrieve top-k chunks
- LLM to answer only from context
- If context does not match, it should refuse

In [15]:
# QA is handled by answer_question(vectordb, question, config=...).

In [16]:
# vectordb is already built/loaded above for FILE_PATH
print("Using indexed file:", FILE_PATH.name, "doc_id:", doc_id[:12])

Using indexed file: 2025-Vistra-Code-of-Conduct.pdf doc_id: a64a3aa5ac56


5. Inspect top-k chunks retrieval

- To confirm the retriever returns relevant snippets and page metadata before generating the final answer

In [17]:
# Inspect retrieved chunks BEFORE generating an answer
question = "What is the smoking policy?"
retriever = vectordb.as_retriever(search_kwargs={"k": TOP_K})
docs = retriever.invoke(question)

print("Retrieved chunks:", len(docs))
print()
print(format_context(docs)[:2000])  # preview first ~2000 chars

Retrieved chunks: 4

--- Snippet 1 (C:\Users\ashle\Project\usecase\data\2025-Vistra-Code-of-Conduct.pdf p.14) ---
privately owned motor vehicle that is parked 
in an area the Company has designated for 
employee parking. 
Q A
Employee Assistance Program
The Company’s Employee Assistance Program is 
designed to confidentially help employees and 
their dependents manage issues with emotional, 
marital, family, or other personal difficulties, 
including dependency on drugs or alcohol. 
Smoking
Smoking is prohibited in all Company buildings, 
facilities, vehicles, and equipment owned or 
leased by the Company unless otherwise 
provided in the Workplace Conduct Policy. This 
includes vaping devices and e-cigarettes. 
Possession of Weapons  
and Firearms
Company policy prohibits the possession of 
weapons, firearms (with or without a license), 
and ammunition whether classified as legal or 
illegal on Company property, including buildings, 
parking lots, recreation facilities, equipment, and

6. Test the question-answer LLM model

In [18]:
question = 'What is the smoking policy?'

out = answer_question(vectordb, question)
print(out["answer"])

The smoking policy prohibits smoking in all Company buildings, facilities, vehicles, and equipment owned or leased by the Company. This prohibition extends to vaping devices and e-cigarettes, unless otherwise specified in the Workplace Conduct Policy [source p.14].


In [19]:
unanswerable = 'What is the company policy on Mars travel reimbursement?'
out2 = answer_question(vectordb, unanswerable)
print(out2['answer'])

The company policy on travel reimbursement states that employees will be reimbursed for actual expenses incurred while conducting company business. These expenses must be business-related and reasonable. Employees are responsible for using discretion when spending company funds, and all expenses should be reviewed and approved in compliance with applicable company policies [source p.30]. 

For specific travel, employees traveling overseas should notify Compliance [source p.30]. However, the provided context does not specifically mention Mars travel reimbursement.


In [20]:
question = 'Provide summary of this document?'

out = answer_question(vectordb, question)
print(out["answer"])

The document is the "2025 Vistra Code of Conduct," which outlines the company's policies and expectations regarding ethical behavior and compliance in various areas. Key sections include:

1. **Workplace Conduct**: Emphasizes fairness, inclusion, health and safety, respect, and prohibits harassment and violence [source p.2].
2. **Use of Company Assets**: Covers intellectual property, confidentiality, data privacy, and records management [source p.2, p.19].
3. **Conflicts of Interest**: Addresses gifts, entertainment, financial interests, and outside activities [source p.2, p.30].
4. **Relationships**: Discusses interactions with customers, suppliers, and competitors, including compliance with laws and regulations [source p.2, p.30].
5. **Other Policies**: Mentions additional policies on endorsements, employee expenses, and international requirements [source p.2, p.30].

The document also provides a compliance helpline for reporting concerns [source p.19].


### Summarizer + QA Demo (Streamlit)

Example Questions:
- Summarize the uploaded content.
- What does the document say about misconduct reporting?
- Is personal device usage allowed on company networks?


In [4]:
# Streamlit runs as a separate app (recommended to launch from a terminal):
#   streamlit run streamlit_app_extend.py

print("Run this in a terminal:")
print("  streamlit run streamlit_app_extend.py")


Run this in a terminal:
  streamlit run streamlit_app_extend.py
